# Lab 5 – The Wumpus 

**Task.** An agent starts at (1,1) of a dark 4x4 cave, facing east. It must grab the gold and leave through (1,1)
with `climb`. 

A pit or the wumpus kills it. 

Rewards: +1000 leaving with the gold, −1000 dying, −1 every action,
−10 for the arrow. 

The agent never perceives its position. It only receives a set of strings
(`stench`, `breeze`, `glitter`, `bump`, `scream`, `at_start`).


Every step has four parts:

1. **Writing the message window**: the percepts plus a few short messages about what is
   around the agent, such as `safe_ahead` (the square in front of the agent is safe and not visited yet) or
   `way_back_left` (the way back is on the left).
2. **The rules choose**: 19 classifiers, each a pair of sets (condition, action). A rule matches when
   all the messages of its condition are in the message window, among the matching rules the highest tier wins.
3. **The cave executes** the action and answers with the new percepts.
4. **Updates the memory**: where the agent believes it is, where it is facing, which squares are visited,
   which squares are proved free of pits or of the wumpus, and the way back.

In [1]:
import random

## 1. Directions and squares

A square is a pair `(row, column)`, counted from 1, with the rows counted from the bottom.

In [2]:
# the cave is a 4x4 grid. A square is a pair (row, column), from 1 to 4
# rows are counted from the bottom: (1, 1) is the start, bottom left
SIZE = 4

# the four directions, in clockwise order
DIRECTIONS = ["N", "E", "S", "W"]

# how a step in each direction changes (row, column)
MOVE = {"N": (1, 0), "E": (0, 1), "S": (-1, 0), "W": (0, -1)}

NAMES = {"N": "north", "E": "east", "S": "south", "W": "west"}


def turn_left(direction):
    # one place back in the list
    index = DIRECTIONS.index(direction)
    return DIRECTIONS[(index + 3) % 4]


def turn_right(direction):
    # one place forward in the list
    index = DIRECTIONS.index(direction)
    return DIRECTIONS[(index + 1) % 4]


def opposite(direction):
    # two places away
    index = DIRECTIONS.index(direction)
    return DIRECTIONS[(index + 2) % 4]


def square_ahead(square, direction):
    # the square next to "square" in "direction", or None if that would be outside the cave (a wall)
    row = square[0] + MOVE[direction][0]
    col = square[1] + MOVE[direction][1]
    if row < 1 or row > SIZE or col < 1 or col > SIZE:
        return None
    return (row, col)


def neighbours(square):
    # the squares next to "square" (not diagonally) that are inside the cave
    result = []
    for direction in DIRECTIONS:
        other = square_ahead(square, direction)
        if other is not None:
            result.append(other)
    return result

## 2. The cave

`cave_from_sketch` turns the drawing into the positions of the pits, the wumpus and the gold. The first line of the
drawing is row 4, so the row number is `4 - i` for the line `i` (counted from 0).

In [3]:
#   .  empty     P  pit     W  wumpus     G  gold
SLIDES_SKETCH = """
...P
WGP.
....
..P.
"""


def cave_from_sketch(sketch):
    # read the drawing and return a dictionary with the pits, the wumpus and the gold
    lines = []
    for line in sketch.split("\n"):
        line = line.strip()
        if line != "":
            lines.append(line)
    cave = {"pits": set(), "wumpus": None, "gold": None}
    for i in range(SIZE):
        row = SIZE - i                  # the first line of the drawing is row 4, the last one is row 1
        for j in range(SIZE):
            col = j + 1                 # characters are counted from 0, columns from 1
            symbol = lines[i][j]
            if symbol == "P":
                cave["pits"].add((row, col))
            elif symbol == "W":
                cave["wumpus"] = (row, col)
            elif symbol == "G":
                cave["gold"] = (row, col)
    return cave

## 3. The world

The world keeps track of the game, meaning info about the cave plus where the agent really is, its score and whether the game is over.
`percepts` tells what the agent perceives where it is, `execute` carries out one action. The agent never reads the
world directly.

In [4]:
#creates a new world from a cave, with the agent starting at (1, 1) facing east, alive, with an arrow and no gold taken
def start_world(cave):
    world = {}
    world["pits"] = cave["pits"]
    world["wumpus"] = cave["wumpus"]
    world["gold"] = cave["gold"]
    world["pos"] = (1, 1)               
    world["facing"] = "E"         
    world["gold_taken"] = False
    world["wumpus_alive"] = True
    world["arrow"] = True
    world["score"] = 0
    world["steps"] = 0
    world["dead"] = False
    world["over"] = False               # true when the agent has died or climbed out
    return world


def percepts(world, bump, scream):
    # what the agent perceives where it is (never its position, only glitter, stench, breeze, bump, scream, at_start)
    result = set()
    here = world["pos"]
    near = neighbours(here)
    if world["wumpus_alive"]:
        if here == world["wumpus"] or world["wumpus"] in near:
            result.add("stench")
    for square in near:
        if square in world["pits"]:
            result.add("breeze")
    if here == world["gold"] and not world["gold_taken"]:
        result.add("glitter")
    if here == (1, 1):
        result.add("at_start")
    if bump:
        result.add("bump")
    if scream:
        result.add("scream")
    return result


def execute(world, action):
    # the cave executes one action and answers with the new percepts
    world["steps"] = world["steps"] + 1
    world["score"] = world["score"] - 1          # every action costs 1
    bump = False
    scream = False
    if action == "turn_l":
        world["facing"] = turn_left(world["facing"])
    elif action == "turn_r":
        world["facing"] = turn_right(world["facing"])
    elif action == "fwd":
        new = square_ahead(world["pos"], world["facing"])
        if new is None:
            bump = True                          # a wall: the agent does not move
        else:
            world["pos"] = new
            if new in world["pits"] or (world["wumpus_alive"] and new == world["wumpus"]):
                world["dead"] = True
                world["over"] = True
                world["score"] = world["score"] - 1000
    elif action == "grab":
        if world["pos"] == world["gold"] and not world["gold_taken"]:
            world["gold_taken"] = True
    elif action == "shoot":
        if world["arrow"]:
            world["arrow"] = False
            world["score"] = world["score"] - 10
            square = world["pos"]
            while True:                          # the arrow flies straight until the wall
                square = square_ahead(square, world["facing"])
                if square is None:
                    break
                if world["wumpus_alive"] and square == world["wumpus"]:
                    world["wumpus_alive"] = False
                    scream = True
    elif action == "climb":
        if world["pos"] == (1, 1):
            world["over"] = True
            if world["gold_taken"]:
                world["score"] = world["score"] + 1000
    return percepts(world, bump, scream)

## 4. The memory

What the agent knows.

* `look_around`: in a square without breeze no neighbour has a pit, in a square without stench no neighbour
  has the wumpus. There is only one wumpus, so it is next to every square that smells: any square that is not
  next to all of them has no wumpus
* `safe_squares`: no pit and no wumpus, and not visited yet
* `update_memory`: after each action, the new heading or position, and the info that tells how to go back

In [5]:
def new_memory():
    memory = {}
    memory["pos"] = (1, 1)              # where the agent believes it is (it knows it starts at (1, 1))
    memory["facing"] = "E"              # facing east
    memory["visited"] = set()           # squares where it has been
    memory["no_pit"] = set()            # squares proved to have no pit
    memory["no_wumpus"] = set()         # squares proved to have no wumpus
    memory["stench"] = set()            # squares where it smelled the wumpus
    memory["back"] = {}                 # square -> direction to go back the way it came
    memory["has_gold"] = False
    memory["arrow"] = True
    return memory


def look_around(memory, seen):
    # the agent is in memory["pos"] and perceives "seen": mark the square and deduce about the neighbours
    here = memory["pos"]
    memory["visited"].add(here)
    for square in neighbours(here):
        if "breeze" not in seen:        # no breeze = no neighbour has a pit
            memory["no_pit"].add(square)
        if "stench" not in seen:        # no stench = no neighbour has the wumpus
            memory["no_wumpus"].add(square)
    if "stench" in seen:
        memory["stench"].add(here)
    if memory["stench"]:                # there is only one wumpus: it is next to every square that smells
        candidates = set.intersection(*[set(neighbours(s)) for s in memory["stench"]])
        for row in range(1, SIZE + 1):
            for col in range(1, SIZE + 1):
                if (row, col) not in candidates:
                    memory["no_wumpus"].add((row, col))


def safe_squares(memory):
    # squares with no pit and no wumpus, not visited yet
    result = set()
    for square in memory["no_pit"]:
        if square in memory["no_wumpus"] and square not in memory["visited"]:
            result.add(square)
    return result


def update_memory(memory, action, seen):
    # after the cave has executed "action" and answered "seen", update what the memory knows about the world
    if action == "turn_l":
        memory["facing"] = turn_left(memory["facing"])
    elif action == "turn_r":
        memory["facing"] = turn_right(memory["facing"])
    elif action == "fwd":
        new = square_ahead(memory["pos"], memory["facing"])
        if new not in memory["visited"]:
            memory["back"][new] = opposite(memory["facing"])
        memory["pos"] = new
        look_around(memory, seen)
    elif action == "grab":
        memory["has_gold"] = True
    elif action == "shoot":
        memory["arrow"] = False
        memory["no_wumpus"].add((1, 2))
        if "scream" in seen:
            memory["no_wumpus"].add((2, 1))

## 5. The message window

The message window is a set of strings, rebuilt at every step. Besides the percepts it says, from the agent's point of view,
where the safe squares and the way back are: `side(facing, direction)` turns north/east/south/west into
ahead/left/right/behind. For example, facing north, a safe square to the east gives `safe_right`.

In [6]:
def side(facing, direction):
    # where "direction" is for an agent that faces "facing"
    if direction == facing:
        return "ahead"
    if direction == turn_left(facing):
        return "left"
    if direction == turn_right(facing):
        return "right"
    return "behind"


def write_messages(memory, seen):
    messages = set()
    for message in seen:                          # the percepts
        messages.add(message)
    safe = safe_squares(memory)
    for direction in DIRECTIONS:                  # finds in which direction there is a safe square
        square = square_ahead(memory["pos"], direction)
        if square in safe:
            messages.add("safe_" + side(memory["facing"], direction))
    if memory["pos"] in memory["back"]:           # on which side is the way back
        crumb = memory["back"][memory["pos"]]
        messages.add("way_back_" + side(memory["facing"], crumb))
    if memory["pos"] == (1, 1):                   # finds if the agent is at home
        messages.add("home_here")
        if "stench" in seen and memory["arrow"] and memory["facing"] == "E":
            messages.add("aim")                      # the start smells: shoot east
    if memory["has_gold"]:                        # if the agent has the gold
        messages.add("has_gold")
    return messages

## 6. The rules

| Tier | Condition → action | Meaning |
|---|---|---|
| 50 | `{glitter} → {grab}` | the gold is here -> grab it |
| 45 | `{aim} → {shoot}` | the start smells -> shoot east |
| 40 | `{has_gold, home_here} → {climb}` | home with the gold -> leave |
| 30 | `{has_gold, way_back_…}` | with the gold -> follow the trail home |
| 20 | `{safe_ahead} → {fwd}` | a safe square ahead -> go |
| 10 | `{safe_left}`, `{safe_right}`, `{safe_behind}` | turn towards a safe square |
| 5 | `{way_back_…}` | nothing safe here -> go back one square |
| 5 | `{home_here} → {climb}` | back home with nothing safe left -> give up |

When several rules of the highest tier match (for example to turn round, left or right), one is chosen at random

In [7]:
def make_rule(name, tier, condition, action):
    rule = {}
    rule["name"] = name
    rule["tier"] = tier
    rule["condition"] = set(condition)
    rule["action"] = set(action)
    return rule


RULES = [
    make_rule("grab", 50, ["glitter"], ["grab"]),
    make_rule("shoot", 45, ["aim"], ["shoot"]),
    make_rule("climb", 40, ["has_gold", "home_here"], ["climb"]),
    make_rule("home_fwd", 30, ["has_gold", "way_back_ahead"], ["fwd"]),
    make_rule("home_left", 30, ["has_gold", "way_back_left"], ["turn_l"]),
    make_rule("home_right", 30, ["has_gold", "way_back_right"], ["turn_r"]),
    make_rule("home_behind_l", 30, ["has_gold", "way_back_behind"], ["turn_l"]),
    make_rule("home_behind_r", 30, ["has_gold", "way_back_behind"], ["turn_r"]),
    make_rule("explore_fwd", 20, ["safe_ahead"], ["fwd"]),
    make_rule("explore_left", 10, ["safe_left"], ["turn_l"]),
    make_rule("explore_right", 10, ["safe_right"], ["turn_r"]),
    make_rule("explore_behind_l", 10, ["safe_behind"], ["turn_l"]),
    make_rule("explore_behind_r", 10, ["safe_behind"], ["turn_r"]),
    make_rule("back_fwd", 5, ["way_back_ahead"], ["fwd"]),
    make_rule("back_left", 5, ["way_back_left"], ["turn_l"]),
    make_rule("back_right", 5, ["way_back_right"], ["turn_r"]),
    make_rule("back_behind_l", 5, ["way_back_behind"], ["turn_l"]),
    make_rule("back_behind_r", 5, ["way_back_behind"], ["turn_r"]),
    make_rule("give_up", 5, ["home_here"], ["climb"]),
]


def choose_rule(messages):
    # the rules whose condition is all in the message window
    matching = []
    for rule in RULES:
        if rule["condition"].issubset(messages):
            matching.append(rule)
    # the highest tier among them
    best_tier = 0
    for rule in matching:
        if rule["tier"] > best_tier:
            best_tier = rule["tier"]
    # the matching rules of that tier. If there are several, one at random
    best = []
    for rule in matching:
        if rule["tier"] == best_tier:
            best.append(rule)
    return random.choice(best)


def action_of(rule):
    for message in rule["action"]:
        return message

## 7. One game

The loop of "play" is the cycle of the agent: messages → rule → action → cave → percepts → memory

In [8]:
def play(cave, show=False):
    world = start_world(cave)
    memory = new_memory()
    seen = percepts(world, False, False)          # what the agent perceives at the start
    look_around(memory, seen)                     # it looks around the start square
    while not world["over"] and world["steps"] < 200:
        messages = write_messages(memory, seen)   #  writes the message window
        rule = choose_rule(messages)              # the rules choose
        action = action_of(rule)
        seen = execute(world, action)             # the cave executes and answers
        update_memory(memory, action, seen)       # updates the memory
        if show:
            print(world["steps"], sorted(messages), "->", rule["name"], "->", action,
                  "| now at", world["pos"], NAMES[world["facing"]], "| perceives", sorted(seen),
                  "| score", world["score"])
    return world


def outcome(world):
    if world["dead"]:
        return "died"
    if world["over"] and world["gold_taken"]:
        return "win"
    if world["over"]:
        return "gave up"
    return "too many steps"

## 8. The cave 

In [9]:
random.seed(1)
cave = cave_from_sketch(SLIDES_SKETCH)
print("pits:", sorted(cave["pits"]), " wumpus:", cave["wumpus"], " gold:", cave["gold"])
world = play(cave, show=True)
print("result:", outcome(world), " score:", world["score"])

pits: [(1, 3), (3, 3), (4, 4)]  wumpus: (3, 1)  gold: (3, 2)
1 ['at_start', 'home_here', 'safe_ahead', 'safe_left'] -> explore_fwd -> fwd | now at (1, 2) east | perceives ['breeze'] | score -1
2 ['breeze', 'way_back_behind'] -> back_behind_l -> turn_l | now at (1, 2) north | perceives ['breeze'] | score -2
3 ['breeze', 'way_back_left'] -> back_left -> turn_l | now at (1, 2) west | perceives ['breeze'] | score -3
4 ['breeze', 'way_back_ahead'] -> back_fwd -> fwd | now at (1, 1) west | perceives ['at_start'] | score -4
5 ['at_start', 'home_here', 'safe_right'] -> explore_right -> turn_r | now at (1, 1) north | perceives ['at_start'] | score -5
6 ['at_start', 'home_here', 'safe_ahead'] -> explore_fwd -> fwd | now at (2, 1) north | perceives ['stench'] | score -6
7 ['safe_right', 'stench', 'way_back_behind'] -> explore_right -> turn_r | now at (2, 1) east | perceives ['stench'] | score -7
8 ['safe_ahead', 'stench', 'way_back_right'] -> explore_fwd -> fwd | now at (2, 2) east | perceives []

## 9. Random caves

Each square but the start has a pit with probability 0.2. The wumpus and the gold are on two different free squares

In [10]:
def random_cave():
    # each square but the start has a pit with probability 0.2
    # the wumpus and the gold go on two different free squares, not on the start
    while True:
        cave = {"pits": set(), "wumpus": None, "gold": None}
        free = []
        for row in range(1, SIZE + 1):
            for col in range(1, SIZE + 1):
                if (row, col) == (1, 1):
                    continue
                if random.random() < 0.2:
                    cave["pits"].add((row, col))
                else:
                    free.append((row, col))
        if len(free) < 2:
            continue
        cave["wumpus"] = random.choice(free)
        free.remove(cave["wumpus"])
        cave["gold"] = random.choice(free)
        return cave


random.seed(100)
counts = {"win": 0, "died": 0, "gave up": 0, "too many steps": 0}
total_score = 0
for game in range(2000):
    world = play(random_cave())
    counts[outcome(world)] = counts[outcome(world)] + 1
    total_score = total_score + world["score"]
print(counts)
print("mean score:", total_score / 2000)

{'win': 802, 'died': 0, 'gave up': 1198, 'too many steps': 0}
mean score: 387.4595
